# SSTrans Training Notebook

This notebook is organized for GitHub usage. Place `github_ready_train.py`, `config.yaml`, `vocab.txt`, model files, tokenizer files, and datasets in the same project directory before running.

## 1. Project setup

In [1]:
from pathlib import Path
import ast
import os
import runpy
import sys

PROJECT_ROOT = Path.cwd()
SCRIPT_PATH = PROJECT_ROOT / "Training_validation_test.py"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(
        "Please put Training_validation_test.py in the same directory as this notebook."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Training script: {SCRIPT_PATH}")

Project root: D:\SSTrans\Article\code\Train
Training script: D:\SSTrans\Article\code\Train\Training_validation_test.py


## 2. Load definitions without executing training

Run this cell when you want to use the classes and functions interactively in the notebook.

In [2]:
source = SCRIPT_PATH.read_text(encoding="utf-8")
tree = ast.parse(source)

tree.body = [
    node
    for node in tree.body
    if not (
        isinstance(node, ast.If)
        and ast.unparse(node.test) == "__name__ == '__main__'"
    )
]

compiled = compile(tree, filename=str(SCRIPT_PATH), mode="exec")
exec(compiled, globals())

print("Definitions loaded.")

Definitions loaded.


## 3. Check runtime environment

In [3]:
import multiprocessing as mp
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU memory: {memory_gb:.2f} GB")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CPU workers: {max(1, mp.cpu_count() - 2)}")

CUDA available: True
GPU device: NVIDIA GeForce RTX 5090
GPU memory: 31.84 GB
Using device: cuda:0
CPU workers: 30


## 4. Load configuration

In [4]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError("config.yaml was not found in the project directory.")

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

features = [
    "solubility",
    "solvation_free_energy",
    "solvation_enthalpy",
    "heat_capacity_cp",
    "heat_capacity_cs",
    "sublimation_enthalpy",
    "polarity_compatibility",
    "size_compatibility",
    "hbond_compatibility",
    "hydrophobicity_compatibility",
    "electrostatic_compatibility",
    "flexibility_compatibility",
    "aromaticity_compatibility",
    "charge_compatibility",
    "solubility_gradient",
]

batch_size = config["batch_size"]
save_path = config["best_model_path"]
save_path_pretrain = config["save_path_pretrain"]
unsupervised_pretrain = config["unsupervised_pretrain"]["pretrained_path"]

setup_seed(config.get("seed", 1))

print(f"Task type: {config['task_type']}")
print(f"Metric type: {config['metric_type']}")
print(f"Batch size: {batch_size}")

Task type: fine_tuning
Metric type: solubility
Batch size: 64


## 5. Initialize tokenizer and data collator

In [5]:
from Mlm_data_collator import SafeDimensionDataCollator
from tokenizer import SMILES_Atomwise_Tokenizer

tokenizer = SMILES_Atomwise_Tokenizer("vocab.txt")
data_collator = SafeDimensionDataCollator(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

print("Tokenizer and collator initialized.")

Tokenizer and collator initialized.


## 6. Run the original training entry point

This cell executes the same training workflow as `python Training_validation_test.py`. Use it when you want the notebook to reproduce the script behavior directly.

In [ ]:
runpy.run_path(str(SCRIPT_PATH), run_name="__main__")

CUDA available: True
GPU device: NVIDIA GeForce RTX 5090
GPU memory: 31.84 GB


D:\SSTrans\Article\code\Train\Training_validation_test.py:600: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1839.)
  self.std = torch.std(tensor)


Loaded existing normalizers
using cuda:0 device.
using 1020310 molecules for training, 3209 molecules for solute extraplation, 1799 molecules for solvent extraplation,1004 molecules for temperature extraplation.
Loading pretrained weights from pretrained_model_2M.pth
Loaded roberta.embeddings.word_embeddings.weight -> embeddings.word_embeddings.weight
Skipping roberta.embeddings.position_embeddings.weight (position_embeddings layer not needed)
Loaded roberta.embeddings.token_type_embeddings.weight -> embeddings.token_type_embeddings.weight
Loaded roberta.embeddings.LayerNorm.weight -> embeddings.LayerNorm.weight
Loaded roberta.embeddings.LayerNorm.bias -> embeddings.LayerNorm.bias
Loaded roberta.encoder.layer.0.attention.self.query.weight -> encoder.layer.0.attention.self.query.weight
Loaded roberta.encoder.layer.0.attention.self.query.bias -> encoder.layer.0.attention.self.query.bias
Loaded roberta.encoder.layer.0.attention.self.key.weight -> encoder.layer.0.attention.self.key.weight


## 7. Optional: create a fully expanded notebook from the script

Run this cell if you want a second notebook where the Python script is split into multiple code cells.

In [ ]:
import json

def make_code_cell(text):
    return {
        "cell_type": "code",
        "metadata": {},
        "source": text.rstrip().splitlines(keepends=True),
        "outputs": [],
        "execution_count": None,
    }

def make_markdown_cell(text):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": text.rstrip().splitlines(keepends=True),
    }

script_source = SCRIPT_PATH.read_text(encoding="utf-8")
script_tree = ast.parse(script_source)
script_lines = script_source.splitlines(keepends=True)

def node_text(node):
    return "".join(script_lines[node.lineno - 1:node.end_lineno])

imports = []
blocks = []

for node in script_tree.body:
    text = node_text(node)
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        imports.append(text)
    elif isinstance(node, ast.ClassDef):
        blocks.append((f"Class: `{node.name}`", text))
    elif isinstance(node, ast.FunctionDef):
        blocks.append((f"Function: `{node.name}`", text))
    else:
        blocks.append(("Execution block", text))

expanded_cells = [make_markdown_cell("# Expanded Training Script Notebook")]

if imports:
    expanded_cells.append(make_markdown_cell("## Imports"))
    expanded_cells.append(make_code_cell("".join(imports)))

for title, text in blocks:
    expanded_cells.append(make_markdown_cell(f"## {title}"))
    expanded_cells.append(make_code_cell(text))

expanded_notebook = {
    "cells": expanded_cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {
            "name": "python",
            "version": "3.x",
            "mimetype": "text/x-python",
            "codemirror_mode": {"name": "ipython", "version": 3},
            "pygments_lexer": "ipython3",
            "nbconvert_exporter": "python",
            "file_extension": ".py",
        },
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

expanded_path = PROJECT_ROOT / "github_ready_train_expanded.ipynb"
expanded_path.write_text(json.dumps(expanded_notebook, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Created: {expanded_path}")